In [ ]:
import requests
import json
import os
import time

from dotenv import load_dotenv

load_dotenv()

True

In [7]:
ig_user_id = os.getenv('INSTAGRAM_BUSINESS_ACCOUNT_ID')
app_id = os.getenv('APP_ID')
app_secret= os.getenv('APP_SECRET')
user_access_token = os.getenv('ACCESS_TOKEN')

long_access_token = os.getenv('LONG_ACESS_TOKEN', False)

In [3]:
print(f'ig_user_id: {ig_user_id}')
print(f'app_id: {app_id}')
print(f'app_secret: {app_secret}')
print(f'user_access_token: {user_access_token}')

ig_user_id: 17841400550881709
app_id: 797255549789768
app_secret: 2039c146eecc9b9eed6638c9bdfd26a1
user_access_token: EAALVGYXXJkgBRD7dp0FxsZBPenaZAPlT3HqRpG7CwpXjP5ZC37fCF6g3dl0gwE42a3AZAxClMkKPYP8bpIRDfaspMux0wBzXUqUtCIAVYS6ZAZAMr6JCBDGKkC1fEorujNZAH5e9aSJiCHsFmUoZAudA6puXXV2VwvoZBxRZCg08k4ZCxE8NKe4p56WZBZC1AAsF5TJb4lcnLL2ZBAmI9g2y4sz8b47CapAok0QJGR3b91EOg0aPYE5aq4NkeshlkN6K5V9sQ9SwIQPTAsmmVTW3tS3qtajzlpmUL7eWiAVQZDZD


# Getting Long Access Token

In [ ]:
if not long_access_token:
    client_id=f'client_id={app_id}'
    client_secret=f'client_secret={app_secret}'
    fb_exchange_token=f'fb_exchange_token={user_access_token}'


    url = f"https://graph.facebook.com/v17.0/oauth/access_token?grant_type=fb_exchange_token&{client_id}&{client_secret}&{fb_exchange_token}"
    response = requests.get(url)
    response_json = response.json()
    print(f"Response JSON: {response_json}")
    long_access_token = response.json()['access_token']
    long_access_token

# Getting Comments

In [13]:
business_discovery_parameters="thumbnail_url,media_type,media_product_type,timestamp,like_count,comments_count,media_url,permalink"
other_parameters="comments{id,from,text,timestamp,like_count,media,parent_id,replies{from,timestamp,like_count,text},hidden,user,username}"
fields=business_discovery_parameters + "," + other_parameters
fields

'thumbnail_url,media_type,media_product_type,timestamp,like_count,comments_count,media_url,permalink,comments{id,from,text,timestamp,like_count,media,parent_id,replies{from,timestamp,like_count,text},hidden,user,username}'

In [14]:
def busca_comentarios(printa=False):
    url=f"https://graph.facebook.com/v25.0/{ig_user_id}/media"
    payload = {
        "fields": fields,
        "access_token": long_access_token,
    }
    response = requests.get(url, params=payload)
    data=response.json()
    if printa:
        print(json.dumps(data, indent=4))
    return data

data = busca_comentarios(printa=True)

{
    "data": [
        {
            "media_type": "CAROUSEL_ALBUM",
            "media_product_type": "FEED",
            "timestamp": "2025-02-19T22:13:21+0000",
            "like_count": 66,
            "comments_count": 8,
            "media_url": "https://scontent.cdninstagram.com/v/t51.75761-15/480412422_18492296773018363_2298370280779625650_n.webp?stp=dst-jpg_e35_tt6&_nc_cat=100&ccb=7-5&_nc_sid=18de74&efg=eyJlZmdfdGFnIjoiQ0FST1VTRUxfSVRFTS5iZXN0X2ltYWdlX3VybGdlbi5DMyJ9&_nc_ohc=i1pCpM3_jl8Q7kNvwGuxe1m&_nc_oc=Adq0QRdAC7tE4dmH_d2YJEqnr0E3QT8ti_rteEUHkZpGKwt-amsB-Mhehz1tXEmiayV0MZkJ3JL3ebGuJlhWMFiL&_nc_zt=23&_nc_ht=scontent.cdninstagram.com&edm=AM6HXa8EAAAA&_nc_gid=pWASD_Bm8kra_slTroJeqQ&oh=00_Af1ipFvks_xLjd3SaQfqPza9NWTJ_gOKJHh4WgZtIiREgA&oe=69E773AE",
            "permalink": "https://www.instagram.com/p/DGRU909S-lj/",
            "comments": {
                "data": [
                    {
                        "id": "18060465086437879",
                        "from": {
    

In [ ]:
banco_comentarios_ja_respondidos = set()

while(True):
    data = busca_comentarios()

    for post in data.get('data', []):
        comments_dict = post.get('comments', {})
        
        for comment in comments_dict.get('data', []):
            comment_id = comment.get('id')

            if comment_id in banco_comentarios_ja_respondidos:
                continue
            
            texto_comentario = comment.get('text', '')

            if 'dica' in texto_comentario.lower():
                commentor_username = comment.get('username')

                print(f"Respondendo ao @{commentor_username}")
    
                reply_url = f"https://graph.facebook.com/v25.0/{comment_id}/replies"

                reply_payload = {
                    "message": f"@{commentor_username} TESTE Obrigado!",
                    "access_token": long_access_token,
                }
                
                reply_response = requests.post(reply_url, params=reply_payload)
                reply_data = reply_response.json()

                banco_comentarios_ja_respondidos.add(comment_id)
                
                print(f"Resposta da API: {reply_response.status_code}")
                
    time.sleep(60)
    

Respondendo ao @eduufiuza
Resposta da API: 200
Respondendo ao @renatapam24
Resposta da API: 200


KeyboardInterrupt: 